In [ ]:
# CASMI26 v4f = v4b (FPNet A engine bank, LB 0.354) + ICEBERG post-ranker isomer re-scoring = the v3.6 model (v1 engine + FPNet A + moderate generation + gated PubChem channel, LB 0.358)
# + fe_v4 feature families computed on the SAME v1 engine run (derivation evidence / class-3 priors, fragmentation 2.0,
# analog-structure relations, DreamsFP views) + a ranker retrained on the v1-engine simulation sim1x with them.
# Built by work/analysis/fe_v4/build_v4v1.py -- edit there.
import os, sys, glob, re, subprocess, time, json, hashlib, pickle
T0 = time.time()
LIB_TAU, REL_TH = 0.9, 600.0
TOPN, ICE_LAM, ICE_BUDGET, ICE_PC = 60, 0.5, 2700, True
SLOTS_AGG, SLOTS_GENTLE = [2, 4, 6, 8, 10], [4, 8, 12, 16, 20]
os.makedirs('/kaggle/working/numba_cache', exist_ok=True)
os.environ['NUMBA_CACHE_DIR'] = '/kaggle/working/numba_cache'
def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]
def sha256(path, n=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(n), b''):
            h.update(b)
    return h.hexdigest()
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
whl = [w for w in glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True) if tag in w]
print('rdkit wheel:', whl)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', whl[0]], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
import rdkit; print('rdkit', rdkit.__version__)
V4CODE = os.path.dirname(os.path.dirname(find('casmi26-v4b-models/**/casmi/engine.py')))
V4MOD = os.path.dirname(find('casmi26-v4b-models/**/MANIFEST.json'))
V3CODE = os.path.dirname(os.path.dirname(find('casmi26-v3-models/**/casmi/engine.py')))
V3MOD = os.path.dirname(find('casmi26-v3-models/**/ranker_0.pkl'))
POOL_DIR = os.path.dirname(find('pool_meta.parquet'))
PC_DIR = os.path.dirname(find('pc_smiles.npy'))
COMP = os.path.dirname(find('test.parquet'))
MAN = json.load(open(os.path.join(V4MOD, 'MANIFEST.json')))
bad = []
for rel, h in sorted(MAN['files'].items()):
    p = os.path.join(V4CODE, rel[len('code/'):]) if rel.startswith('code/') else os.path.join(V4MOD, rel)
    if not os.path.exists(p) or sha256(p) != h:
        bad.append(rel)
assert not bad, f'dataset files differ from MANIFEST.json: {bad}'
print('V4CODE', V4CODE, '\nV4MOD', V4MOD, '\nV3CODE', V3CODE, '\nPOOL', POOL_DIR, '\nPC', PC_DIR, '\nCOMP', COMP)
print(f"{len(MAN['files'])} files match MANIFEST.json; families {MAN['families']}", f'{time.time()-T0:.0f}s')

In [ ]:
# v1 engine code (the 0.358 model's engine) + fe_v4; pool symlinks; spectrum cache from train.parquet
sys.path.insert(0, V4CODE)
os.makedirs('work', exist_ok=True)
for f in ['pool_meta.parquet', 'pool_fp.npy', 'pool_frag_off.npy', 'pool_frag_mass.npy', 'fp_bits.npy', 'train_structs.parquet', 'train_fp_sel.npy']:
    src = os.path.join(POOL_DIR, f)
    dst = f'work/{f}'
    if not os.path.exists(dst):
        os.symlink(src, dst)
from casmi.build import build_spec_cache
build_spec_cache(os.path.join(COMP, 'train.parquet'), 'work')
import casmi; print('casmi from', casmi.__file__)
sys.path.insert(1, os.path.join(V4CODE, 'fe_v4'))
print(f'cache built {time.time()-T0:.0f}s')

In [ ]:
# v1 engine exactly as in v3.6 (FPNet A, EngineCfg(generate=True)) wrapped by V1FE (recording hooks + families)
import numpy as np, pandas as pd, torch, lightgbm as lgb
from casmi.library import Library
from casmi.pool import Pool
from casmi.engine import Engine, EngineCfg, FEATURES
from casmi import fpnet, chem
import v1engine
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
L = Library('work'); P = Pool('work')
tfp = np.load('work/train_fp_sel.npy')
bank = fpnet.ModelBank([os.path.join(V4MOD, 'fpnet_0.pt')], device=dev)
E = Engine(L, P, tfp, EngineCfg(generate=True), bank)
V = v1engine.V1FE(E, [os.path.join(V4MOD, m['file']) for m in MAN['fe_models']], MAN['families'], device=dev)
if 'derivation' in MAN['families']:
    n_bad = V.mods['derivation'].self_test()
    print('derivation prior self-test mismatches:', n_bad)
    assert n_bad == 0, 'RDKit build differs from the one the derivation priors were built with'
RK = pickle.load(open(os.path.join(V4MOD, MAN['ranker']['file']), 'rb'))
RK_FEATS = list(RK['features']); RK_B = [lgb.Booster(model_str=s) for s in RK['boosters']]
def rank_score(X, names):
    pos = {n: i for i, n in enumerate(names)}
    miss = [f for f in RK_FEATS if f not in pos]
    assert not miss, f'ranker features not produced (would be zero-filled): {miss}'
    Xs = np.ascontiguousarray(X[:, [pos[f] for f in RK_FEATS]], dtype=np.float32)
    return np.mean([b.predict(Xs, num_threads=4) for b in RK_B], axis=0)
print('engine ready; ranker features', len(RK_FEATS), 'boosters', len(RK_B), f'{time.time()-T0:.0f}s')

In [ ]:
# ---- EXPERIMENT HARNESS (not a submission) ----
# Held-out timsTOF natural products (enveda-np-examples) run through the v1 engine + v4b ranker under three simulations:
#   c1: only this molecule's enveda-np-examples spectra hidden (other libraries still have it)   -> class 1
#   c2: every spectrum of the structure hidden, structure stays in the pool                    -> class 2
#   c3: c2 + structure dropped from the pool (can only be recovered by generation)              -> class 3
# Caveat: FPNet / ranker / fe_v4 resources were likely trained on these structures, so c2/c3 are optimistic.
import time, pickle
NPLIB, MAXMOL = 9, int(os.environ.get('MAXMOL', 250))
meta = L.meta
ok = (L.lib == NPLIB) & meta.adduct.isin(chem.TEST_ADDUCTS).values & (L.sid >= 0) & (L.n_clean > 0)
rows_all = np.flatnonzero(ok)
sids = np.unique(L.sid[rows_all])[:MAXMOL]
print('val structures', len(sids), 'spectra', len(rows_all), flush=True)

def spectra_of(rows):
    out = []
    for r in rows[:16]:
        a, b = L.off[r], L.off[r + 1]
        ce = float(meta.ce_mean.values[r])
        out.append(dict(mz=L.mz[a:b].astype(np.float64), it=L.it[a:b].astype(np.float64), mode=int(L.mode[r]),
                        adduct=meta.adduct.values[r], prec=float(L.prec[r]), ce=ce if np.isfinite(ce) else np.nan,
                        ce_n=int(max(1, meta.ce_n.values[r]))))
    return out

res, lists = [], {}
t1 = time.time()
for i, sid in enumerate(sids):
    rows = rows_all[L.sid[rows_all] == sid]
    spectra = spectra_of(rows)
    nm = meta.nm.values[rows]; nm = nm[np.isfinite(nm)]
    if not len(nm): continue
    target = float(np.median(nm)); truth = L.struct_key[sid]
    for scen in ('c1', 'c2', 'c3'):
        ex = np.zeros(len(L.sid), bool)
        if scen == 'c1':
            ex[rows] = True; kw = dict(exclude=ex, exclude_sid=int(sid), exclude_lib=NPLIB)
        else:
            ex[L.sid == sid] = True; kw = dict(exclude=ex, exclude_sid=int(sid))
            if scen == 'c3': kw['drop_pid'] = int(P.key2pid.get(truth, -1))
        t2 = time.time()
        try:
            C, X, F, names, info = V.run(spectra, target, **kw)
            keys, scs = [], []
            if len(C.pid):
                score = rank_score(np.concatenate([X, F], 1), list(FEATURES) + list(names))
                seen = set()
                for j in np.argsort(-score, kind='stable'):
                    if C.key[j] in seen: continue
                    seen.add(C.key[j]); keys.append(C.key[j]); scs.append(float(score[j]))
                    if len(keys) >= TOPN: break
            rank = keys.index(truth) + 1 if truth in keys else 0
            res.append(dict(sid=int(sid), scen=scen, rank=rank, rr=(1 / rank if 0 < rank <= 25 else 0.0),
                            in_cands=truth in set(C.key), n_cand=len(C.pid), lib_max=float(info.get('lib_max', 0.0)),
                            secs=time.time() - t2))
            lists[(int(sid), scen)] = (keys, scs, float(info.get('lib_max', 0.0)))
        except Exception as e:
            print('ERROR', sid, scen, repr(e)); res.append(dict(sid=int(sid), scen=scen, rank=0, rr=0.0, err=repr(e)))
    if (i + 1) % 10 == 0:
        R = pd.DataFrame(res)
        print(f'{i+1}/{len(sids)} {time.time()-t1:.0f}s', R.groupby('scen').rr.mean().round(3).to_dict(), flush=True)

R = pd.DataFrame(res)
R.to_csv('/kaggle/working/harness.csv', index=False)
pickle.dump(lists, open('/kaggle/working/harness_lists.pkl', 'wb'))
print('\n==== MRR@25 by simulated class ====')
print(R.groupby('scen').agg(mrr=('rr', 'mean'), top1=('rank', lambda r: (r == 1).mean()),
                            recall25=('rank', lambda r: ((r > 0) & (r <= 25)).mean()),
                            in_cands=('in_cands', 'mean'), secs=('secs', 'mean'), n=('rr', 'size')).round(3))
print('total', f'{time.time()-T0:.0f}s')
